In [8]:
import nltk
import re

from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from gensim.models import Word2Vec

In [2]:
nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package gutenberg to /usr/share/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading p

True

In [3]:
# Load raw text
hamlet_text = gutenberg.raw('shakespeare-hamlet.txt')

# Load Hamlet as words
hamlet_words = gutenberg.words('shakespeare-hamlet.txt')

print("Total words in Hamlet:", len(hamlet_words))

print("\nFirst 500 characters of Hamlet:")
print(hamlet_text[:500])

Total words in Hamlet: 37360

First 500 characters of Hamlet:
[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not


In [4]:

# Convert text to lowercase
text = hamlet_text.lower()

# Remove punctuation and special characters
text = re.sub(r'[^a-z\s]', ' ', text)

# Tokenize the text into words
tokens = word_tokenize(text)

# Load English stopwords
stop_words = set(stopwords.words('english'))

# Remove stopwords
cleaned_tokens = [
    word for word in tokens
    if word not in stop_words
    and len(word) > 1
]

print("\nNumber of tokens after preprocessing:", len(cleaned_tokens))

print("\nFirst 30 preprocessed words:")
print(cleaned_tokens[:30])


Number of tokens after preprocessing: 15834

First 30 preprocessed words:
['tragedie', 'hamlet', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'two', 'centinels', 'barnardo', 'fran', 'nay', 'answer', 'stand', 'vnfold', 'selfe', 'bar', 'long', 'liue', 'king', 'fran', 'barnardo', 'bar', 'fran', 'come', 'carefully']


In [5]:

sentences = nltk.sent_tokenize(hamlet_text.lower())

processed_sentences = []

for sentence in sentences:

    # Remove punctuation, numbers, and special characters
    sentence = re.sub(r'[^a-z\s]', ' ', sentence)

    # Tokenize sentence
    words = word_tokenize(sentence)

    # Remove stopwords and very short words
    words = [
        word for word in words
        if word not in stop_words
        and len(word) > 1
    ]

    # Add only non-empty sentences
    if len(words) > 0:
        processed_sentences.append(words)


print("\nNumber of processed sentences:", len(processed_sentences))

print("\nExample processed sentence:")
print(processed_sentences[0])




Number of processed sentences: 2321

Example processed sentence:
['tragedie', 'hamlet', 'william', 'shakespeare', 'actus', 'primus']


In [6]:
# Assign grammatical tags to each word
pos_tags = nltk.pos_tag(cleaned_tokens)

print("\nFirst 20 POS-tagged words:")
print(pos_tags[:20])


First 20 POS-tagged words:
[('tragedie', 'JJ'), ('hamlet', 'NN'), ('william', 'NN'), ('shakespeare', 'NN'), ('actus', 'NN'), ('primus', 'NN'), ('scoena', 'NN'), ('prima', 'NN'), ('enter', 'NN'), ('barnardo', 'VBP'), ('francisco', 'NN'), ('two', 'CD'), ('centinels', 'NNS'), ('barnardo', 'VBP'), ('fran', 'JJ'), ('nay', 'RB'), ('answer', 'VBP'), ('stand', 'VBP'), ('vnfold', 'JJ'), ('selfe', 'JJ')]


In [9]:
# Create Porter Stemmer object
stemmer = PorterStemmer()

# Apply stemming to every cleaned token
stemmed_words = [
    stemmer.stem(word)
    for word in cleaned_tokens
]

print("\nFirst 30 stemmed words:")
print(stemmed_words[:30])




First 30 stemmed words:
['tragedi', 'hamlet', 'william', 'shakespear', 'actu', 'primu', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'two', 'centinel', 'barnardo', 'fran', 'nay', 'answer', 'stand', 'vnfold', 'self', 'bar', 'long', 'liue', 'king', 'fran', 'barnardo', 'bar', 'fran', 'come', 'care']


In [10]:
# Create WordNet Lemmatizer object
lemmatizer = WordNetLemmatizer()


# Function to convert POS tags to WordNet format
def get_wordnet_pos(treebank_tag):

    if treebank_tag.startswith('J'):
        return 'a'       # Adjective

    elif treebank_tag.startswith('V'):
        return 'v'       # Verb

    elif treebank_tag.startswith('N'):
        return 'n'       # Noun

    elif treebank_tag.startswith('R'):
        return 'r'       # Adverb

    else:
        return 'n'

In [11]:
# Perform POS-aware lemmatization
lemmatized_words = []

for word, tag in pos_tags:

    wordnet_pos = get_wordnet_pos(tag)

    lemma = lemmatizer.lemmatize(
        word,
        pos=wordnet_pos
    )

    lemmatized_words.append(lemma)


print("\nFirst 30 lemmatized words:")
print(lemmatized_words[:30])


First 30 lemmatized words:
['tragedie', 'hamlet', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'francisco', 'two', 'centinels', 'barnardo', 'fran', 'nay', 'answer', 'stand', 'vnfold', 'selfe', 'bar', 'long', 'liue', 'king', 'fran', 'barnardo', 'bar', 'fran', 'come', 'carefully']


In [13]:
# Split raw text into sentences
sentences = nltk.sent_tokenize(
    hamlet_text.lower()
)

# List to store processed sentences
processed_sentences = []


# Process one sentence at a time
for sentence in sentences:

    # Remove punctuation, numbers, and special characters
    sentence = re.sub(
        r'[^a-z\s]',
        ' ',
        sentence
    )

    # Tokenize the sentence
    words = word_tokenize(sentence)

    # Remove stopwords and short words
    words = [
        word for word in words
        if word not in stop_words
        and len(word) > 1
    ]

    # Perform POS tagging for the sentence
    tagged_words = nltk.pos_tag(words)

    # Lemmatize each word using its POS tag
    lemmatized_sentence = []

    for word, tag in tagged_words:

        wordnet_pos = get_wordnet_pos(tag)

        lemma = lemmatizer.lemmatize(
            word,
            pos=wordnet_pos
        )

        lemmatized_sentence.append(lemma)

    # Add non-empty processed sentences
    if len(lemmatized_sentence) > 0:

        processed_sentences.append(
            lemmatized_sentence
        )


print("\nNumber of processed sentences:")
print(len(processed_sentences))


Number of processed sentences:
2321


In [14]:
print("\nExample processed sentence:")
print(processed_sentences[10])


Example processed sentence:
['come', 'carefully', 'vpon', 'houre', 'bar']


In [15]:
word2vec_model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,     # Dimension of each word vector
    window=5,            # Context window size
    min_count=1,         # Include words appearing at least once
    workers=4,
    sg=1                 # 1 = Skip-Gram, 0 = CBOW
)


In [16]:
print("\nVocabulary size:")
print(len(word2vec_model.wv))

print("\nFirst 20 words in vocabulary:")
print(list(word2vec_model.wv.index_to_key[:20]))



Vocabulary size:
4174

First 20 words in vocabulary:
['ham', 'lord', 'king', 'haue', 'come', 'shall', 'thou', 'let', 'hamlet', 'good', 'make', 'hor', 'thy', 'enter', 'like', 'well', 'oh', 'go', 'know', 'would']


In [17]:
word = "hamlet"

if word in word2vec_model.wv:
    vector = word2vec_model.wv[word]

    print(f"\nEmbedding vector for '{word}':")
    print(vector)

    print("\nVector shape:")
    print(vector.shape)

else:
    print(f"\n'{word}' is not present in the vocabulary.")



Embedding vector for 'hamlet':
[-0.04598338  0.11866543  0.05749025 -0.03966507  0.0540852  -0.24731456
  0.06133996  0.2869248  -0.0196029  -0.1108722  -0.12235057 -0.28297195
  0.00484923  0.00552481  0.07240367 -0.11169866 -0.02154769 -0.17180316
 -0.0075942  -0.31107062  0.10795584  0.15066582 -0.01198466 -0.03363268
 -0.08810648  0.0768318  -0.08422383 -0.06659164 -0.21813011  0.03343244
  0.07635572 -0.00562466  0.02094003 -0.00380447 -0.07908741  0.23295662
 -0.02887746 -0.11004414 -0.1339591  -0.2172828   0.05983776 -0.1850228
 -0.06902239  0.02963397  0.13850015 -0.13145195 -0.06045331 -0.06532588
  0.04327157  0.06163596  0.02462468 -0.18394512 -0.04861492 -0.03966461
 -0.16629875  0.12255419  0.08476628 -0.1031203  -0.11876643  0.05971875
  0.09369816  0.09711441 -0.03364512  0.07621334 -0.07052524  0.13530494
 -0.02076229  0.05580077 -0.22142114  0.14581923 -0.04184303  0.06205602
  0.2036019  -0.02886068  0.13892016  0.1549492  -0.05252136 -0.03465374
 -0.07885545  0.0424

In [18]:
word = "king"

if word in word2vec_model.wv:

    similar_words = word2vec_model.wv.most_similar(
        word,
        topn=10
    )

    print(f"\nWords most similar to '{word}':")

    for similar_word, similarity in similar_words:
        print(f"{similar_word}: {similarity:.4f}")


Words most similar to 'king':
life: 0.9987
time: 0.9986
th: 0.9985
may: 0.9984
haue: 0.9984
hath: 0.9984
shall: 0.9983
come: 0.9983
much: 0.9983
thou: 0.9983


In [19]:
word1 = "king"
word2 = "queen"

if word1 in word2vec_model.wv and word2 in word2vec_model.wv:

    similarity = word2vec_model.wv.similarity(
        word1,
        word2
    )

    print(
        f"\nSimilarity between '{word1}' and '{word2}': "
        f"{similarity:.4f}"
    )




Similarity between 'king' and 'queen': 0.9378
